In [1]:
import scanpy as sc
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

import scipy.stats as stats
from scipy.stats import spearmanr, hypergeom

from statsmodels.stats.multitest import multipletests

# GO enrichment
import gseapy as gp

## CellOracle
import celloracle as co

In [2]:
## DATA
adata = sc.read_h5ad("../data/data_diff_express_lncRNA.h5ad")
base_GRN = pd.read_parquet('base_GRN_edge_list.parquet')

## CellOracle inference

In [ ]:
# =============================================================
# STEP 1: PREPARE THE ANNDATA FOR CELLORACLE
# CellOracle needs raw counts or log-normalized in adata.X,
# NOT the Z-scored matrix. We restore from adata.raw.
# =============================================================

# CellOracle internally uses adata.X for regression.
# We restore the log-normalized matrix (not Z-scored) from adata.raw
# so that Ridge Regression operates on meaningful expression magnitudes.
adata_co = adata.copy()
adata_co.X = adata.raw[:, adata.var_names].X  # filtrar raw a los mismos genes

# CellOracle requires the UMAP coordinates already stored
# Verify they exist
print("Embeddings available:", list(adata_co.obsm.keys()))
# Should contain 'X_umap'

print("Clusters:", adata_co.obs['leiden'].value_counts())

# =============================================================
# CONVERT EDGE LIST TO CELLORACLE TFdict FORMAT
# TFdict structure: {target_gene: [TF1, TF2, ...]}
# In your edge list: source = regulator, target = target gene
# =============================================================

# Load your extended edge list (with RMST1 edges already added)
extended_GRN = pd.read_parquet("../data/extended_GRN_edge_list.parquet")

# Filter to genes present in adata to avoid ghost nodes in the network
genes_in_adata = set(adata_co.var_names)

extended_GRN_filtered = extended_GRN[
    extended_GRN['target'].isin(genes_in_adata) &
    extended_GRN['source'].isin(genes_in_adata)
]

print(f"Edges before filtering to adata genes: {len(extended_GRN)}")
print(f"Edges after filtering: {len(extended_GRN_filtered)}")

# Convert to TFdict: group regulators by target gene
TFdict = (
    extended_GRN_filtered
    .groupby('target')['source']
    .apply(list)
    .to_dict()
)

print(f"Target genes with at least one regulator: {len(TFdict)}")

# Sanity check: verify RMST1 edges are present
rmst1_name = 'Rmst'
rmst1_as_regulator = {
    target: regs for target, regs in TFdict.items()
    if rmst1_name in regs
}
print(f"Genes regulated by {rmst1_name} in TFdict: {len(rmst1_as_regulator)}")


# =============================================================
# INITIALIZE ORACLE AND INJECT TFdict DIRECTLY
# =============================================================

oracle = co.Oracle()

oracle.import_anndata_as_raw_count(
    adata=adata_co,
    cluster_column_name='leiden',
    embedding_name='X_umap'
)

oracle.perform_PCA()
oracle.knn_imputation(n_pca_dims=19, k=20)

# Assign TFdict directly — bypasses import_TF_data entirely
oracle.TFdict = TFdict

print("TFdict assigned. Proceeding to fit GRN.")


# =============================================================
# STEP 5: FIT GRN — RIDGE REGRESSION PER CLUSTER
# This estimates the actual regulatory weights W_ij from
# expression co-variation, using oracle.TFdict as sparsity mask.
# alpha controls Ridge penalization (higher = sparser weights)
# =============================================================

oracle.fit_GRN_for_simulation(
    alpha=10,                          # Ridge regularization strength
    use_cluster_specific_TFdict=False  # use same prior for all clusters
)


# =============================================================
# STEP 6: SIMULATE RMST1 KNOCKOUT
# Sets Rmst expression to 0 in all cells and propagates
# the perturbation through the fitted GRN.
# =============================================================

oracle.simulate_shift(
    perturb_condition={"Rmst": 0.0},  # KO: set Rmst to zero
    n_propagation=3   # number of propagation steps through the network
)


# =============================================================
# STEP 7: COMPUTE TRANSITION PROBABILITIES AND EMBEDDING SHIFT
# Translates the gene expression shift into a probability of
# transitioning to neighboring cells in the UMAP embedding.
# =============================================================

oracle.estimate_transition_prob(
    n_neighbors=40,
    knn_random=True,
    sampled_fraction=0.5
)

oracle.calculate_embedding_shift(sigma_corr=0.05)


# =============================================================
# STEP 8: VISUALIZE VECTOR FIELD ON UMAP
# =============================================================

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Grid-based vector field (cleaner visualization)
oracle.plot_simulation_flow_on_grid(
    scale=0.4,
    ax=axes[0],
    color_by='leiden'
)
axes[0].set_title('RMST1 KO — vector field (grid)')

# Single-cell arrows
oracle.plot_simulation_flow_random_sampling(
    scale=0.4,
    ax=axes[1],
    color_by='leiden',
    n_each_cluster=30
)
axes[1].set_title('RMST1 KO — vector field (single cells)')

plt.tight_layout()
#plt.savefig('../figures/rmst1_ko_vector_field.pdf', dpi=150)
plt.show()


# =============================================================
# SAVE ORACLE OBJECT FOR FURTHER ANALYSIS
# =============================================================

#oracle.to_hdf5("../data/oracle_rmst1_ko.celloracle.hdf5")

Embeddings available: ['X_pca', 'X_umap']
Clusters: 0    379
1    336
2    328
3    323
5    152
6     86
Name: leiden, dtype: int64
Edges before filtering to adata genes: 6876578
Edges after filtering: 1737222
Target genes with at least one regulator: 8900
Genes regulated by Rmst in TFdict: 0
11145 genes were found in the adata. Note that Celloracle is intended to use around 1000-3000 genes, so the behavior with this number of genes may differ from what is expected.
TFdict assigned. Proceeding to fit GRN.


  0%|          | 0/6 [00:00<?, ?it/s]

ValueError: Gene Rmst is not included in the Gene expression matrix.

Nanog example: we don't have Rmst included yet

In [18]:
# =============================================================
# STEP 6: SIMULATE RMST1 KNOCKOUT
# Sets Rmst expression to 0 in all cells and propagates
# the perturbation through the fitted GRN.
# =============================================================

oracle.simulate_shift(
    perturb_condition={"Nanog": 0.0},  # KO: set Rmst to zero
    n_propagation=3   # number of propagation steps through the network
)


# =============================================================
# STEP 7: COMPUTE TRANSITION PROBABILITIES AND EMBEDDING SHIFT
# Translates the gene expression shift into a probability of
# transitioning to neighboring cells in the UMAP embedding.
# =============================================================

oracle.estimate_transition_prob(
    n_neighbors=40,
    knn_random=True,
    sampled_fraction=0.5
)

oracle.calculate_embedding_shift(sigma_corr=0.05)

In [19]:
# =============================================================
# STEP 8: VISUALIZE VECTOR FIELD ON UMAP
# =============================================================

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Grid-based vector field (cleaner visualization)
oracle.plot_simulation_flow_on_grid(
    scale=0.4,
    ax=axes[0],
)
axes[0].set_title('Nanog KO — vector field (grid)')

# Single-cell arrows with random sampling
oracle.plot_simulation_flow_random_on_grid(
    scale=0.4,
    ax=axes[1],
    color='leiden',           # ← usar 'color', no 'color_by'
    n_each_cluster=30
)
axes[1].set_title('Nanog KO — vector field (single cells)')

plt.tight_layout()
# plt.savefig('../figures/Nanog_ko_vector_field.pdf', dpi=150)
plt.show()

AttributeError: 'Oracle' object has no attribute 'flow'